In [ ]:
!pip install awscli==1.29.62 botocore==1.31.62 urllib3==1.26.16

In [ ]:
!aws s3 cp s3://spacenet-dataset/spacenet/SN2_buildings/tarballs/ . --recursive --no-sign-request

download: s3://spacenet-dataset/spacenet/SN2_buildings/tarballs/AOI_5_Khartoum_Test_public.tar.gz to ./AOI_5_Khartoum_Test_public.tar.gz
download: s3://spacenet-dataset/spacenet/SN2_buildings/tarballs/AOI_3_Paris_Test_public.tar.gz to ./AOI_3_Paris_Test_public.tar.gz
download: s3://spacenet-dataset/spacenet/SN2_buildings/tarballs/SN2_buildings_train_AOI_3_Paris.tar.gz to ./SN2_buildings_train_AOI_3_Paris.tar.gz
download: s3://spacenet-dataset/spacenet/SN2_buildings/tarballs/AOI_4_Shanghai_Test_public.tar.gz to ./AOI_4_Shanghai_Test_public.tar.gz
download: s3://spacenet-dataset/spacenet/SN2_buildings/tarballs/AOI_2_Vegas_Test_public.tar.gz to ./AOI_2_Vegas_Test_public.tar.gz
download: s3://spacenet-dataset/spacenet/SN2_buildings/tarballs/SN2_buildings_train_AOI_5_Khartoum.tar.gz to ./SN2_buildings_train_AOI_5_Khartoum.tar.gz
download failed: s3://spacenet-dataset/spacenet/SN2_buildings/tarballs/SN2_buildings_train_AOI_2_Vegas.tar.gz to ./SN2_buildings_train_AOI_2_Vegas.tar.gz [Errno 28]

In [ ]:
!ls

AOI_2_Vegas_Test_public.tar.gz	   sample_data
AOI_3_Paris_Test_public.tar.gz	   SN2_buildings_train_AOI_3_Paris.tar.gz
AOI_4_Shanghai_Test_public.tar.gz  SN2_buildings_train_AOI_5_Khartoum.tar.gz
AOI_5_Khartoum_Test_public.tar.gz


In [ ]:
!tar -xvf SN2_buildings_train_AOI_3_Paris.tar.gz

Streaming output truncated to the last 5000 lines.
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img1468.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img1021.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img921.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img689.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img260.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img600.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img813.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img1164.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img1501.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img164.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img1142.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img1347.tif
AOI_3_Paris_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_3_Paris_img1773.tif
AOI_3_Paris_Train/R

In [ ]:
!ls AOI_3_Paris_Train

geojson  MUL  MUL-PanSharpen  PAN  RGB-PanSharpen  summaryData


In [ ]:
!pip install rasterio geopandas shapely

In [ ]:
import os
import numpy as np
import rasterio
import geopandas as gpd
from rasterio.features import rasterize
from tqdm import tqdm

image_dir = "AOI_3_Paris_Train/RGB-PanSharpen"
geojson_dir = "AOI_3_Paris_Train/geojson"
mask_dir = "AOI_3_Paris_Train/masks"

os.makedirs(mask_dir, exist_ok=True)

image_files = os.listdir(image_dir)

for img_file in tqdm(image_files):

    if not img_file.endswith(".tif"):
        continue

    img_path = os.path.join(image_dir, img_file)
    geojson_path = os.path.join(
        geojson_dir,
        img_file.replace("RGB-PanSharpen_", "buildings_").replace(".tif", ".geojson")
    )

    if not os.path.exists(geojson_path):
        continue

    with rasterio.open(img_path) as src:
        image = src.read()
        transform = src.transform
        out_shape = (src.height, src.width)

    gdf = gpd.read_file(geojson_path)

    mask = rasterize(
        [(geom, 1) for geom in gdf.geometry],
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype=np.uint8
    )

    mask_path = os.path.join(mask_dir, img_file.replace(".tif", ".png"))

    rasterio.open(
        mask_path,
        "w",
        driver="PNG",
        height=mask.shape[0],
        width=mask.shape[1],
        count=1,
        dtype=mask.dtype
    ).write(mask, 1)

print("Mask generation complete.")

100%|██████████| 1148/1148 [00:00<00:00, 130302.30it/s]

Mask generation complete.


In [ ]:
!ls AOI_3_Paris_Train/masks

In [ ]:
!ls AOI_3_Paris_Train/RGB-PanSharpen | head

RGB-PanSharpen_AOI_3_Paris_img1000.tif
RGB-PanSharpen_AOI_3_Paris_img1001.tif
RGB-PanSharpen_AOI_3_Paris_img1003.tif
RGB-PanSharpen_AOI_3_Paris_img1007.tif
RGB-PanSharpen_AOI_3_Paris_img1008.tif
RGB-PanSharpen_AOI_3_Paris_img1009.tif
RGB-PanSharpen_AOI_3_Paris_img100.tif
RGB-PanSharpen_AOI_3_Paris_img1010.tif
RGB-PanSharpen_AOI_3_Paris_img1011.tif
RGB-PanSharpen_AOI_3_Paris_img1012.tif


In [ ]:
!ls AOI_3_Paris_Train/geojson/buildings | head

buildings_AOI_3_Paris_img1000.geojson
buildings_AOI_3_Paris_img1001.geojson
buildings_AOI_3_Paris_img1003.geojson
buildings_AOI_3_Paris_img1007.geojson
buildings_AOI_3_Paris_img1008.geojson
buildings_AOI_3_Paris_img1009.geojson
buildings_AOI_3_Paris_img100.geojson
buildings_AOI_3_Paris_img1010.geojson
buildings_AOI_3_Paris_img1011.geojson
buildings_AOI_3_Paris_img1012.geojson


In [ ]:
import os
import numpy as np
import rasterio
import geopandas as gpd
from rasterio.features import rasterize
from tqdm import tqdm

image_dir = "AOI_3_Paris_Train/RGB-PanSharpen"
geojson_dir = "AOI_3_Paris_Train/geojson/buildings"
mask_dir = "AOI_3_Paris_Train/masks"

os.makedirs(mask_dir, exist_ok=True)

image_files = [f for f in os.listdir(image_dir) if f.endswith(".tif")]
print("Total images:", len(image_files))

for img_file in tqdm(image_files):

    img_id = img_file.replace("RGB-PanSharpen_", "").replace(".tif", "")
    geojson_file = f"buildings_{img_id}.geojson"

    geojson_path = os.path.join(geojson_dir, geojson_file)
    img_path = os.path.join(image_dir, img_file)

    if not os.path.exists(geojson_path):
        continue

    with rasterio.open(img_path) as src:
        transform = src.transform
        out_shape = (src.height, src.width)

    gdf = gpd.read_file(geojson_path)

    mask = rasterize(
        [(geom, 1) for geom in gdf.geometry],
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype=np.uint8
    )

    mask_path = os.path.join(mask_dir, img_file.replace(".tif", ".png"))

    with rasterio.open(
        mask_path,
        "w",
        driver="PNG",
        height=mask.shape[0],
        width=mask.shape[1],
        count=1,
        dtype=mask.dtype
    ) as dst:
        dst.write(mask, 1)

print("Mask generation complete.")

Total images: 1148


  0%|          | 0/1148 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
  0%|          | 1/1148 [00:00<08:21,  2.29it/s]/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The i

Mask generation complete.


In [ ]:
!ls AOI_3_Paris_Train/masks | head

RGB-PanSharpen_AOI_3_Paris_img1000.png
RGB-PanSharpen_AOI_3_Paris_img1001.png
RGB-PanSharpen_AOI_3_Paris_img1003.png
RGB-PanSharpen_AOI_3_Paris_img1007.png
RGB-PanSharpen_AOI_3_Paris_img1008.png
RGB-PanSharpen_AOI_3_Paris_img1009.png
RGB-PanSharpen_AOI_3_Paris_img100.png
RGB-PanSharpen_AOI_3_Paris_img1010.png
RGB-PanSharpen_AOI_3_Paris_img1011.png
RGB-PanSharpen_AOI_3_Paris_img1012.png


In [ ]:
!tar -xvf SN2_buildings_train_AOI_5_Khartoum.tar.gz
!tar -xvf SN2_buildings_train_AOI_4_Shanghai.tar.gz

Streaming output truncated to the last 5000 lines.
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img51.tif
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img912.tif
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img174.tif
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img114.tif
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img1085.tif
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img1214.tif
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img336.tif
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img1130.tif
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img209.tif
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img1671.tif
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img510.tif
AOI_5_Khartoum_Train/RGB-PanSharpen/RGB-PanSharpen_AOI_5_Khartoum_img830.tif
AOI_5_Khartoum_Train/R

In [ ]:
import os
import numpy as np
import rasterio
import geopandas as gpd
from rasterio.features import rasterize
from tqdm import tqdm

aoi = "AOI_5_Khartoum_Train"

image_dir = os.path.join(aoi, "RGB-PanSharpen")
geojson_dir = os.path.join(aoi, "geojson", "buildings")
mask_dir = os.path.join(aoi, "masks")

os.makedirs(mask_dir, exist_ok=True)

image_files = [f for f in os.listdir(image_dir) if f.endswith(".tif")]
print("Total images:", len(image_files))

for img_file in tqdm(image_files):

    img_id = img_file.replace("RGB-PanSharpen_", "").replace(".tif", "")
    geojson_file = f"buildings_{img_id}.geojson"

    geojson_path = os.path.join(geojson_dir, geojson_file)
    img_path = os.path.join(image_dir, img_file)

    if not os.path.exists(geojson_path):
        continue

    with rasterio.open(img_path) as src:
        transform = src.transform
        out_shape = (src.height, src.width)

    gdf = gpd.read_file(geojson_path)

    mask = rasterize(
        [(geom, 1) for geom in gdf.geometry],
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype=np.uint8
    )

    mask_path = os.path.join(mask_dir, img_file.replace(".tif", ".png"))

    with rasterio.open(
        mask_path,
        "w",
        driver="PNG",
        height=mask.shape[0],
        width=mask.shape[1],
        count=1,
        dtype=mask.dtype
    ) as dst:
        dst.write(mask, 1)

print("Khartoum mask generation complete.")

Total images: 1012


  0%|          | 0/1012 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
  0%|          | 2/1012 [00:00<00:55, 18.29it/s]/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The i

Khartoum mask generation complete.


In [ ]:
!ls AOI_5_Khartoum_Train/masks | head

RGB-PanSharpen_AOI_5_Khartoum_img1001.png
RGB-PanSharpen_AOI_5_Khartoum_img1003.png
RGB-PanSharpen_AOI_5_Khartoum_img1004.png
RGB-PanSharpen_AOI_5_Khartoum_img1007.png
RGB-PanSharpen_AOI_5_Khartoum_img1008.png
RGB-PanSharpen_AOI_5_Khartoum_img100.png
RGB-PanSharpen_AOI_5_Khartoum_img1010.png
RGB-PanSharpen_AOI_5_Khartoum_img1011.png
RGB-PanSharpen_AOI_5_Khartoum_img1012.png
RGB-PanSharpen_AOI_5_Khartoum_img1013.png


In [ ]:
import os
os.makedirs("dataset/images", exist_ok=True)
os.makedirs("dataset/masks", exist_ok=True)

In [ ]:
import os
import cv2
import rasterio
import numpy as np
from tqdm import tqdm

aois = ["AOI_3_Paris_Train", "AOI_5_Khartoum_Train"]

target_size = 256
img_count = 0

for aoi in aois:

    image_dir = os.path.join(aoi, "RGB-PanSharpen")
    mask_dir = os.path.join(aoi, "masks")

    image_files = [f for f in os.listdir(image_dir) if f.endswith(".tif")]

    print(f"Processing {aoi} ...")

    for img_file in tqdm(image_files):

        img_path = os.path.join(image_dir, img_file)
        mask_path = os.path.join(mask_dir, img_file.replace(".tif", ".png"))

        if not os.path.exists(mask_path):
            continue

        # Read image
        with rasterio.open(img_path) as src:
            image = src.read([1,2,3])
            image = np.transpose(image, (1,2,0))

        mask = cv2.imread(mask_path, 0)

        # Normalize image to 0–255
        image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
        image = image.astype(np.uint8)

        # Resize
        image_resized = cv2.resize(image, (target_size, target_size))
        mask_resized = cv2.resize(mask, (target_size, target_size))

        # Save
        cv2.imwrite(f"dataset/images/img_{img_count}.png", image_resized)
        cv2.imwrite(f"dataset/masks/img_{img_count}.png", mask_resized)

        img_count += 1

print("Total processed images:", img_count)

Processing AOI_3_Paris_Train ...


100%|██████████| 1148/1148 [00:24<00:00, 46.80it/s]


Processing AOI_5_Khartoum_Train ...


100%|██████████| 1012/1012 [00:20<00:00, 48.46it/s]

Total processed images: 2160


In [ ]:
!ls dataset/images | head

img_0.png
img_1000.png
img_1001.png
img_1002.png
img_1003.png
img_1004.png
img_1005.png
img_1006.png
img_1007.png
img_1008.png


In [ ]:
!rm -rf AOI_3_Paris_Train
!rm -rf AOI_5_Khartoum_Train
!rm *.tar.gz

In [ ]:
!pip install segmentation-models-pytorch --quiet

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class BuildingDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths = image_paths
        self.mask_paths = mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image / 255.0

        mask = cv2.imread(self.mask_paths[idx], 0)
        mask = mask / 255.0
        mask = np.expand_dims(mask, axis=0)

        image = np.transpose(image, (2, 0, 1))

        return torch.tensor(image, dtype=torch.float32), \
               torch.tensor(mask, dtype=torch.float32)

In [ ]:
image_dir = "dataset/images"
mask_dir = "dataset/masks"

images = sorted([os.path.join(image_dir, f) for f in os.listdir(image_dir)])
masks = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)])

train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    images, masks, test_size=0.2, random_state=42
)

train_dataset = BuildingDataset(train_imgs, train_masks)
val_dataset = BuildingDataset(val_imgs, val_masks)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [ ]:
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(device)

loss_fn = smp.losses.DiceLoss(mode="binary")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            val_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}")

Epoch 1/10 | Train Loss: 0.7512 | Val Loss: 0.5191
Epoch 2/10 | Train Loss: 0.4887 | Val Loss: 0.4933
Epoch 3/10 | Train Loss: 0.4419 | Val Loss: 0.4643
Epoch 4/10 | Train Loss: 0.4044 | Val Loss: 0.4112
Epoch 5/10 | Train Loss: 0.3925 | Val Loss: 0.3567
Epoch 6/10 | Train Loss: 0.3545 | Val Loss: 0.5139
Epoch 7/10 | Train Loss: 0.3560 | Val Loss: 0.3339
Epoch 8/10 | Train Loss: 0.3402 | Val Loss: 0.3595
Epoch 9/10 | Train Loss: 0.3316 | Val Loss: 0.2937
Epoch 10/10 | Train Loss: 0.3345 | Val Loss: 0.3057


In [ ]:
torch.save(model.state_dict(), "building_unet_base.pth")

In [ ]:
from google.colab import files
files.download("building_unet_base.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>